# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:   
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [5]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 220749b1


In [6]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [7]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [8]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [9]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [10]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/risheeksomu/AIE9/07_Deep_Agents/workspace


In [11]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] comprehensive_morning_routine_guide.md (19019 bytes)
[FILE] comprehensive_stress_management_guide.md (10026 bytes)
[FILE] morning_energy_routine_guide.md (34069 bytes)
[FILE] morning_routine_guide.md (20973 bytes)
[FILE] personalized_sleep_improvement_plan.md (6156 bytes)
[DIR] research
[FILE] science_backed_morning_routine_guide.md (3429 bytes)
[FILE] sleep_improvement_research_report.md (12403 bytes)
[FILE] stress_management_comprehensive_guide.md (8665 bytes)


In [12]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[DIR] morning_routines
[FILE] sleep_notes.md (242 bytes)


In [13]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [14]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/risheeksomu/AIE9/07_Deep_Agents/workspace


In [15]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Perfect! I've successfully created your personalized sleep improvement plan and saved it to `/your_personalized_sleep_improvement_plan.md`. 

## Summary of Your Plan

Your sleep improvement plan is designed as a progressive 8-week program that addresses your specific issues:

**🎯 Week 1-2: Choose Your Foundation**
- **Option A (Recommended):** Fix your wake time at 7:00 AM every day, regardless of when you went to bed
- **Option B:** Remove your phone from the bedroom entirely

**🌅 Week 3-4: Add Circadian Optimization**
- Morning light therapy within 30 minutes of waking
- Evening light dimming 2 hours before bed
- Gradual bedtime advancement (15 minutes every 2-3 days)

**🛏️ Week 5-8: Advanced Sleep Quality**
- Optimize sleep environment (60-67°F, complete darkness)
- Develop a 30-45 minute pre-sleep routine
- Fine-tune timing using 90-minute sleep cycles

## Key Insights from Research

1. **Your issues are interconnected** - inconsistent bedtime creates "social jet la

In [16]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Research sleep hygiene principles (completed)
✅ [todo_7] Save sleep plan to file (completed)
✅ [todo_6] Research circadian rhythm regulation techniques (completed)
✅ [todo_8] Research blue light and digital device impacts (completed)
✅ [todo_10] Research consistent sleep schedule strategies (completed)
✅ [todo_12] Research sleep quality improvement methods (completed)
✅ [todo_14] Research progressive implementation approaches (completed)
✅ [todo_16] Compile comprehensive sleep improvement summary (completed)


Workspace contents:
  [FILE] comprehensive_morning_routine_guide.md (19019 bytes)
  [FILE] comprehensive_stress_management_guide.md (10026 bytes)
  [FILE] morning_energy_routine_guide.md (34069 bytes)
  [FILE] morning_routine_guide.md (20973 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6156 bytes)
  [DIR] resea

---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:

Todo lists introduce overhead that's only justified for genuinely complex, multi-step tasks. For simple queries like "What's the recommended daily water intake?", creating a todo list adds unnecessary latency and token costs without improving the response quality-the agent could answer directly from its knowledge. The trade-off is clear: explicit planning becomes worthwhile when tasks involve multiple phases, require context management across turns, or benefit from visible progress tracking, but becomes counterproductive for straightforward questions that can be answered in a single LLM call.

Todo items should maintain medium-level granularity that provides enough direction for autonomous execution without being overly prescriptive. Items that are too broad (e.g., "Complete wellness assessment") leave the agent without actionable guidance, while extremely specific todos (e.g., "Ask user their bedtime," "Ask user their wake time") create unnecessary overhead and clutter the context. The optimal approach involves 4-8 medium-granularity items per complex task-specific enough to track meaningful progress milestones, yet broad enough that each todo represents a substantive chunk of work the agent can independently complete.

Unfinished todos introduce "context rot" where the agent wastes resources attempting to complete stale or irrelevant tasks, potentially entering loops where it continuously tries to address todos that are no longer applicable. This risk can be mitigated through practical safeguards: implementing maximum todo limits (e.g., 10-15 items per session), adding timeouts that auto-complete or archive old todos, and designing agents to periodically review and prune the todo list based on current context. Additionally, agents should be prompted to mark todos as "blocked" or "cancelled" rather than leaving them perpetually pending, maintaining a clean working state that prevents efficiency degradation.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
The 16KB health document should be stored in the file management system rather than loaded into the system prompt on every turn. This allows the agent to use targeted retrieval-reading only relevant sections when answering specific prompts-instead of burning 16,000 tokens of context on every interaction. Similarly, user metrics tracked over time (e.g., daily weight, sleep hours, exercise logs) should be stored in structured files that the agent can query when analyzing trends or generating reports. This approach keeps the conversation context lean while preserving access to complete historical data through on-demand file reads.

User conditions like allergies and medications are safety-critical and should be stored in long-term memory (LangGraph Store) with a dedicated namespace, NOT in files. This ensures the information persists across sessions and can be quickly retrieved at the start of every conversation through a simple memory lookup. While this data could technically be stored in files, memory systems provide faster access, structured querying, and are specifically designed for frequently-accessed metadata. The agent's system prompt should include explicit instructions to check the safety profile before making any dietary, supplement, or exercise recommendations, creating a mandatory safety checkpoint in the workflow.

Certain information must remain in the system prompt and should never be offloaded: (1) core behavioral guidelines (e.g., "Always provide medical disclaimers," "Never diagnose conditions"), (2) the agent's fundamental identity and capabilities (e.g., "You are a wellness assistant with access to planning and file tools"), and (3) critical safety rules (e.g., "If user mentions severe symptoms, recommend professional consultation"). These elements define the agent's foundational behavior and must be immediately accessible for every response-offloading them to files or memory would introduce dangerous latency where the agent might make recommendations before loading safety constraints. Additionally, the current user's goal and active task context should remain in the conversation to maintain coherent, context-aware interactions.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [ ]:
@tool
def read_health_guide(path: str) -> str:
    """Read from the HealthWellnessGuide.txt file in the data folder.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = Path("data/HealthWellnessGuide.txt")
    if not target.exists():
        return f"File not found: {target}"
    return target.read_text()

workspace_path = Path("workspace")
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

research_tools = [write_todos, update_todo, list_todos, read_health_guide]

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=research_tools,
    backend=filesystem_backend,
    system_prompt="""You are a wellness research specialist. Your job is to:

Your workflow:
1. Create a todo list for the research process
2. Use read_health_guide to find relevant information
3. Extract and organize findings
4. Save a comprehensive markdown report to a file
5. Update todo status as you complete tasks

Be thorough but concise. Structure your output clearly with headers and bullet points."""
)

print("Research agent created!")


TODO_STORE.clear()

# Run the research task
result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Research agent created!
Agent response:
Perfect! I've successfully completed your stress management research and created a comprehensive guide. Here's what I've delivered:

## **Comprehensive Stress Management Guide: Key Highlights**

### **Evidence-Based Strategies Included:**

**Immediate Relief Techniques (5 minutes or less):**
1. **Box Breathing** - 4-4-4-4 method for instant calm
2. **5-4-3-2-1 Grounding** - Sensory technique to redirect focus
3. **Progressive Muscle Relaxation** - Quick tension/release method
4. **Cold Water Technique** - Activates vagus nerve

**Long-Term Foundational Strategies:**
1. **Regular Exercise** - 150+ minutes weekly, reduces cortisol
2. **Mindfulness & Meditation** - 5-20 minutes daily, 35% anxiety reduction
3. **Sleep Optimization** - Evidence-based sleep hygiene practices
4. **Social Connection** - Building supportive relationships
5. **Time Management** - Boundary setting and prioritization

**Advanced Science-Backed Techniques:**
1. **Expressive W

In [18]:
# Check todos
print("\n" + "="*60)
print("TODO STATUS:")
print("="*60)
print(list_todos.invoke({}))

# Check generated files
print("\n" + "="*60)
print("GENERATED FILES:")
print("="*60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")


TODO STATUS:
✅ [todo_1] Read health guide for stress management information (completed)
✅ [todo_3] Research evidence-based stress management techniques (completed)
✅ [todo_5] Organize findings into categories (completed)
✅ [todo_7] Create comprehensive markdown guide (completed)
✅ [todo_9] Save final report to file (completed)
✅ [todo_6] Research Advanced Stress Management Techniques (completed)
✅ [todo_8] Compile Scientific Evidence (completed)
✅ [todo_10] Create Implementation Guides (completed)
✅ [todo_12] Document Benefits and Timeframes (completed)
✅ [todo_14] Structure Comprehensive Summary (completed)

GENERATED FILES:
  [FILE] advanced_stress_management_research.md (14704 bytes)
  [FILE] comprehensive-stress-management-guide.md (12716 bytes)
  [FILE] comprehensive_morning_routine_guide.md (19019 bytes)
  [FILE] comprehensive_stress_management_guide.md (10026 bytes)
  [FILE] morning_energy_routine_guide.md (34069 bytes)
  [FILE] morning_routine_guide.md (20973 bytes)
  [FILE] p

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [24]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [25]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [ ]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

In [26]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Create comprehensive morning routine guide (completed)
✅ [todo_5] Save the guide as a markdown file (completed)

Generated files in workspace:
  [FILE] comprehensive_morning_routine_guide.md (18997 bytes)
  [FILE] comprehensive_stress_management_guide.md (10026 bytes)
  [FILE] morning_routine_guide.md (20973 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6156 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research_report.md (12403 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [20]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [21]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [22]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [23]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! Great to connect with you again. I can see from your profile that your primary goal is to improve energy levels with better sleep as a secondary goal, and I know you prefer morning workouts and detailed information. Given your mild anxiety, I'll recommend exercises that can help with both energy and stress management.

Here's a personalized morning exercise routine for you:

## **Alex's Energy-Boosting Morning Routine (30-40 minutes)**

### **Monday, Wednesday, Friday - Energizing Cardio Mix**
1. **Warm-up (5 minutes)**
   - Light stretching and arm circles
   - Gentle marching in place

2. **Main workout (25 minutes)**
   - 10 minutes moderate-intensity walking/jogging
   - 10 minutes bodyweight circuit (jumping jacks, squats, push-ups, lunges)
   - 5 minutes yoga flow (sun salutations work great for morning energy)

3. **Cool-down (5 minutes)**
   - Deep breathing exercises (helps with anxiety management)
   - Light stretching

### **Tuesday, Thursday - Stren

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [24]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [25]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [26]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [27]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [28]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Based on your wellness assessment, here are my personalized recommendations:

## Immediate Action Items (Start Today)

1. **Set a Sleep Schedule**
   - Go to bed and wake up at the same time daily (aim for 7-8 hours)
   - Create a 30-minute wind-down routine before bed (no screens, dim lighting)

2. **Implement Hourly Movement**
   - Set phone/computer reminders to stand and move for 2-3 minutes every hour
   - Take stairs instead of elevators when possible

3. **Track Your Current Habits**
   - Log sleep times and quality for one week
   - Note energy levels throughout the day
   - Record current eating patterns

## Short-term Goals (1-2 Weeks)

1. **Establish Exercise Routine**
   - Start with 20-30 minute walks during lunch breaks
   - Add 2-3 bodyweight exercise sessions per week (can be done at home)

2. **Optimize Your Workspace**
   - Adjust monitor height to reduce neck strain
   - Consider a standing desk converter or stability ball chair
   - Keep healthy vege

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [38]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents-cli 0.0.20


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [30]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [31]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [32]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect! 🎉 Your comprehensive 2-week wellness program is complete! Here's what I've created for you:

## 📁 Your Wellness Program Files:

1. **📋 Master Overview:** `/alex_2week_wellness_program_overview.md`
   - Complete 14-day schedule
   - Daily checklists and quick reference tools
   - Progress tracking templates

2. **🏃‍♂️ Exercise Plan:** `/alex_2week_exercise_plan.md`
   - 3x/week, 30-minute morning workouts
   - Progressive difficulty over 2 weeks
   - Designed for building consistency and energy

3. **🥗 Nutrition Plan:** `/alex_2week_vegetarian_nutrition_plan.md`
   - Complete vegetarian meal plans with recipes
   - Grocery lists and meal prep tips
   - Energy-boosting, anxiety-supporting foods

4. **🧘‍♂️ Stress & Sleep Program:** `/alex_2week_stress_sleep_program.md`
   - Daily stress management routines
   - Complete evening wind-down protocols
   - Workplace stress techniques

## 🌟 Key Program Highlights:

- **Personalized for YOU:** Considers your ve

In [33]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create Exercise Program (completed)
✅ [todo_3] Create Vegetarian Nutrition Plan (completed)
✅ [todo_5] Create Stress Management & Sleep Program (completed)
✅ [todo_7] Save Exercise Plan to File (completed)
✅ [todo_9] Save Nutrition Plan to File (completed)
✅ [todo_11] Save Stress & Sleep Plan to File (completed)
✅ [todo_13] Create Master Wellness Program Overview (completed)

GENERATED FILES
  [FILE] advanced_stress_management_research.md (14704 bytes)
  [FILE] alex_2week_exercise_plan.md (3322 bytes)
  [FILE] alex_2week_stress_sleep_program.md (8509 bytes)
  [FILE] alex_2week_vegetarian_nutrition_plan.md (5653 bytes)
  [FILE] alex_2week_wellness_program_overview.md (8261 bytes)
  [FILE] comprehensive-stress-management-guide.md (12716 bytes)
  [FILE] comprehensive_morning_routine_guide.md (19019 bytes)
  [FILE] comprehensive_stress_management_guide.md (10026 bytes)
  [DIR] exercise_programs/
  [FILE] morning_energy_routine_guide.md (34069 bytes)
  [FILE] mo

In [34]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of stress_management_sleep_optimization_program_for_alex.md:
# 2-Week Stress Management and Sleep Optimization Program for Alex

## Overview
This comprehensive plan focuses on managing work-related stress, improving sleep quality, and addressing mild anxiety through practical techniques integrated into a working professional's schedule.

### Key Components
1. Daily Stress Management Techniques
2. Evening Routines for Better Sleep
3. Mindfulness and Relaxation Practices
4. Strategies for Work-Life Balance
5. Addressing the Connection Between Stress, Anxiety, and Sleep Quality

---

## Week 1

### Day 1 - Monday
**Daily Stress Management:**  
- **Morning Exercise:** 20-minute walk or jog to boost endorphins.  
- **Work Breaks:** Use the Pomodoro technique (25 mins work, 5 mins break) to minimize overwhelm.  
  
**Evening Routine:**  
- **Dinner:** Light meal at least 2 hours before bed.  
- **Relaxation:** Spend 15 mins reading a book or listening to soft music.
- **Sleep Prepa

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
The decision to share tools versus assigning distinct tools depends on whether the tools provide domain-agnostic capabilities or specialized functions. Domain-agnostic tools like file system operations (read_file, write_file, ls) should be shared across all subagents since these are fundamental capabilities needed regardless of specialization; the wellness coach example demonstrates this by having all subagents inherit file tools from the FilesystemBackend. However, highly specialized tools should be assigned distinctly to prevent inappropriate access and maintain clear domain boundaries. For instance, a nutrition specialist should have exclusive access to calculate_macros and analyze_meal_plan tools, while an exercise specialist would have distinct tools like calculate_heart_rate_zones and design_workout_program. This separation serves dual purposes: it prevents subagents from attempting tasks outside their expertise, and it creates security boundaries where sensitive operations are restricted to authorized subagents only.

Choosing the appropriate model for each subagent requires balancing task complexity against cost constraints, recognizing that not all delegation requires the most powerful models. The coordinator agent should typically use expensive, high-capability models like Claude Sonnet because it handles complex orchestration, decision-making, and synthesis of multiple subagent outputs. Research and data extraction subagents can effectively use cheaper models like GPT-4o-mini or Claude Haiku since their tasks involve straightforward information retrieval and pattern matching rather than deep reasoning. Writing and synthesis subagents should use more expensive models because their outputs are user-facing and quality directly impacts the final deliverable. This tiered approach allows the system to process high volumes of simpler tasks cheaply while reserving expensive compute for complex reasoning and final outputs

The optimal level of subagent specialization creates coherent domains that are substantial enough to justify delegation overhead while maintaining clear boundaries that minimize cross-agent dependencies. Subagents that are too broadly defined (such as a single "health agent" handling all wellness tasks) fail to provide meaningful specialization and essentially recreate the problems Deep Agents are designed to solve. Conversely, hyper-specific subagents (like separate agents for calorie counting, macro calculation, and meal suggestion) create excessive coordination overhead where the main agent spends more resources managing handoffs than the subagents save through specialization. The sweet spot typically involves three to five subagents with clear, coherent domains such as exercise specialist, nutrition specialist, and mindfulness specialist in the wellness coach example. Each specialist should encompass enough scope that they can independently complete meaningful chunks of work and produce clean outputs with minimal back-and-forth communication.


## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
Production wellness applications require content filtering and hard stops to prevent harmful advice, such as blocking recommendations to discontinue prescribed medications or immediately escalating emergencies like chest pain to professional care. The InMemoryStore and local file system used in development must be replaced with PostgreSQL for long-term memory and cloud storage like S3 for user files, ensuring data survives server restarts and scales beyond a single machine. Medical disclaimers must appear on every health-related response to manage liability, and the system prompt should include explicit constraints against diagnosing conditions or replacing professional medical advice.

Supporting multiple concurrent users requires authentication, namespace isolation in the memory store (each user gets their own (user_id, "profile") namespace), and file system permissions preventing cross-user data access to comply with HIPAA and GDPR requirements. Production systems need comprehensive logging of agent actions, error tracking with tools like Sentry, and LangSmith tracing to debug complex multi-agent workflows when things go wrong. Without observability, troubleshooting production issues becomes guesswork; you need to see which subagent failed, what tools were called, and where the agent deviated from expected behavior.

Subagent architectures can quickly become expensive when serving many users, requiring rate limits per user (for example, 50 requests per day), token budgets per query (capping at 100k tokens), and strategic caching of common queries to avoid redundant API calls. Model selection should be tiered based on usage patterns: free-tier users get cheaper models like GPT-4o-mini, while paid users access Claude Sonnet for higher quality responses. Monitoring API spending in real time with alerts at specific thresholds prevents runaway costs where a single bug or adversarial user could drain the budget overnight.

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [39]:
### YOUR CODE HERE ###

# Step 1: Define your subagent configurations

# Subagent 1: Exercise Specialist
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise programming and fitness. Use for creating workout plans and exercise recommendations.",
    "system_prompt": """You are an exercise specialist focused on the 30-day wellness challenge.

Your responsibilities:
- Design safe, progressive exercise routines
- Adapt plans based on user fitness level
- Track exercise completion and progress
- Provide form cues and safety tips

Always consider user's current fitness level and any physical limitations.
Create achievable daily exercise goals.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini"  # Cost-effective for structured planning
}

# Subagent 2: Nutrition Specialist
nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition and meal planning. Use for dietary advice and meal suggestions.",
    "system_prompt": """You are a nutrition specialist focused on the 30-day wellness challenge.

Your responsibilities:
- Create balanced, sustainable meal suggestions
- Track daily nutrition goals
- Adapt plans based on user preferences and restrictions
- Provide simple, practical meal ideas

Always respect dietary restrictions and preferences.
Focus on sustainable habits, not extreme diets.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini"  # Cost-effective for pattern-based tasks
}

# Subagent 3: Mindfulness Specialist (Optional but recommended)
mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management and daily motivation. Use for mindfulness practices and encouragement.",
    "system_prompt": """You are a mindfulness and motivation specialist for the 30-day wellness challenge.

Your responsibilities:
- Suggest daily mindfulness practices (meditation, breathing, gratitude)
- Provide motivational check-ins
- Track engagement and adapt practices
- Celebrate progress and milestones

Keep practices simple and achievable (5-10 minutes).
Be encouraging and supportive.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini"  # Sufficient for motivational content
}

print("Subagent configurations defined!")

# Step 2: Create any additional tools you need


# Step 3: Build the main coordinator agent


# Step 4: Test with a user creating their 30-day challenge


# Step 5: Simulate a daily check-in and adaptation


Subagent configurations defined!


In [40]:
from langchain_core.tools import tool
from datetime import datetime

# Step 2: Create any additional tools you need
@tool
def log_daily_checkin(user_id: str, day_number: int, notes: str) -> str:
    """Log a daily check-in for the wellness challenge.
    
    Args:
        user_id: User's unique identifier
        day_number: Which day of the challenge (1-30)
        notes: User's notes about how the day went
    
    Returns:
        Confirmation message
    """
    checkin_namespace = (user_id, "checkins")
    checkin_key = f"day_{day_number}"
    
    checkin_data = {
        "day": day_number,
        "date": datetime.now().isoformat(),
        "notes": notes
    }
    
    memory_store.put(checkin_namespace, checkin_key, checkin_data)
    return f"Logged check-in for day {day_number}"

@tool
def get_challenge_progress(user_id: str) -> str:
    """Get the user's progress in the 30-day challenge.
    
    Args:
        user_id: User's unique identifier
    
    Returns:
        Summary of completed days and progress
    """
    checkin_namespace = (user_id, "checkins")
    items = list(memory_store.search(checkin_namespace))
    
    if not items:
        return "No check-ins logged yet. Start your challenge!"
    
    days_completed = len(items)
    result = [f"Challenge Progress: {days_completed}/30 days completed"]
    
    # Show recent check-ins
    recent = sorted(items, key=lambda x: x.key, reverse=True)[:5]
    result.append("\nRecent check-ins:")
    for item in recent:
        day = item.value.get("day", "?")
        notes = item.value.get("notes", "")[:50]  # Truncate long notes
        result.append(f"  Day {day}: {notes}...")
    
    return "\n".join(result)

print("Additional tools defined!")

Additional tools defined!


In [ ]:
# Step 3: Build the main coordinator agent

# Use the same workspace and backend
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True
)

# Combine all tools
wellness_coach_tools = [
    # Planning
    write_todos,
    update_todo,
    list_todos,
    # Memory
    get_user_profile,
    save_user_preference,
    # Challenge-specific
    log_daily_checkin,
    get_challenge_progress,
]

# Create the 30-day wellness coach
wellness_coach_30day = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=wellness_coach_tools,
    backend=filesystem_backend,
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach managing 30-day wellness challenges.

## Your Role
Help users successfully complete personalized 30-day wellness challenges covering:
- Exercise (consistent movement habits)
- Nutrition (sustainable eating patterns)  
- Mindfulness (stress management and mental wellbeing)

## Workflow for New Challenges
1. Check user profile to understand their goals and baseline
2. Create a weekly todo structure (4 weeks of todos, not 30 daily ones)
3. Delegate to specialists:
   - exercise-specialist: Create progressive workout plan
   - nutrition-specialist: Design sustainable meal approach
   - mindfulness-specialist: Suggest daily practices
4. Save the complete 30-day plan to a file
5. Track progress through daily check-ins

## Daily Check-ins
- Ask how yesterday went
- Log feedback using log_daily_checkin
- Adapt recommendations based on feedback
- Update todo status weekly
- Celebrate milestones (day 7, 14, 21, 30)

## Important Guidelines
- Create 4 weekly todos, not 30 daily ones (manageable tracking)
- Each week should build on the previous week
- Be encouraging and realistic
- Adapt to user feedback
- Save all plans to files for user reference

Use subagents for specialized content, coordinate the overall experience."""
)

print("30-Day Wellness Coach created with all 4 Deep Agent elements!")
print(f"- Planning: {len([t for t in wellness_coach_tools if 'todo' in t.name])} todo tools")
print(f"- Memory: {len([t for t in wellness_coach_tools if 'profile' in t.name or 'checkin' in t.name])} memory tools")
print(f"- Subagents: {len([exercise_specialist, nutrition_specialist, mindfulness_specialist])} specialists")
print(f"- Files: {workspace_path}") 

30-Day Wellness Coach created with all 4 Deep Agent elements!
- Planning: 3 todo tools
- Memory: 2 memory tools
- Subagents: 3 specialists
- Files: /Users/risheeksomu/AIE9/07_Deep_Agents/workspace


In [45]:
# Step 4: Test with a user creating their 30-day challenge

# Reset for clean demo
TODO_STORE.clear()

# Create the challenge
result = wellness_coach_30day.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. 

I want to start a 30-day wellness challenge focusing on:
1. Building an exercise habit (I'm a beginner, can do 20 mins 3x/week)
2. Eating more vegetables (I'm vegetarian)
3. Managing work stress better

Please create a comprehensive 30-day plan for me with weekly goals."""
    }]
})

print("="*60)
print("COACH RESPONSE:")
print("="*60)
print(result["messages"][-1].content)

COACH RESPONSE:
## 🎉 Your Comprehensive 30-Day Wellness Challenge is Ready!

Alex, I've created a detailed, personalized plan that addresses all your goals while considering your preferences and circumstances. Here's what I've set up for you:

### **Your Challenge Structure:**
✅ **4 Weekly Goals** (not overwhelming daily tasks)
✅ **Progressive Exercise Plan** - Starting gentle, building to 20-min workouts 3x/week
✅ **Vegetarian-Friendly Nutrition** - Focused on increasing vegetables systematically  
✅ **Stress Management Toolkit** - From 5-minute practices to workplace techniques

### **Key Highlights:**
- **Week 1:** Foundation building with basic bodyweight exercises, 1 extra veggie serving daily, and 5-minute stress practices
- **Week 2:** Consistency focus with slight increases and 10-minute mindfulness
- **Week 3:** Progressive challenges with new movements, colorful veggie combos, and workplace stress techniques  
- **Week 4:** Mastery and creating sustainable long-term patterns


In [46]:
# Step 5: Simulate a daily check-in and adaptation

# Simulate end of day 3
followup = wellness_coach_30day.invoke({
    "messages": result["messages"] + [{
        "role": "user",
        "content": """Day 3 check-in (user_alex):

Exercise: ✅ Did a 20-min walk - felt good!
Nutrition: ⚠️ Only had veggies at dinner - forgot during the day
Mindfulness: ✅ 5-min breathing exercise before bed

The exercise felt easy. Can I increase intensity?
Need help remembering to eat veggies throughout the day."""
    }]
})

print("\n" + "="*60)
print("ADAPTATION RESPONSE:")
print("="*60)
print(followup["messages"][-1].content)


ADAPTATION RESPONSE:
Great job on Day 3, Alex! 🌟 I love that you're already feeling good about the exercise and staying consistent with mindfulness. Let me address both of your concerns:

## 🏃‍♂️ **Exercise Intensity**
Since the walk felt easy, let's make a small adjustment while staying in Week 1's foundation phase:

**Tomorrow's Exercise Options:**
- **Option A:** Add 5 minutes of bodyweight exercises after your walk (try the Week 1 Workout 1 from your plan)
- **Option B:** Turn your walk into intervals: alternate 2 minutes normal pace + 1 minute brisk pace
- **Option C:** Add some gentle hills or stairs to your walking route

*Remember: We want to build the habit first, then intensity. Week 2 will naturally increase the challenge!*

## 🥕 **Vegetable Memory Helpers**

Here are some practical strategies to help you remember veggies throughout the day:

### **Immediate Solutions:**
1. **Phone Reminders:** Set 2 alarms - "Lunch veggie check" and "Snack time veggies"
2. **Visual Cues:**

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)